# Testing strategy

The profiler's quality bar is unusual for a data tool: every claim it
emits must be *checkable*, so the tests check the claims the same way a
skeptical reader would — against planted ground truth, adversarial
fixtures, and controls that must fail.

## Planted structure, asserted recovery

Unit and integration tests plant a table with known structure and
assert the report contains the matching statements — and only those.
A fixture with three FD violations asserts the approximate dependency
with exactly three violations; a decision-table fixture asserts the
cross-validated accuracy against a hand-computable baseline; a regimes
fixture asserts the recovered slopes and the breakpoint. Negative
assertions matter as much: a near-majority target must yield no PRISM
rules, a unique column must suppress the "X → everything" flood, a
failed rule evaluation must drop the candidate.

The taxonomy has its own tests: equivalent statements from different
detectors consolidate into one finding without pooling counts; different
directions, conditions, and forms stay distinct; selection limits apply
across detectors; a custom detector emitting an existing category is
consolidated like a built-in.

## Adversarial regressions

The interesting tests encode past mistakes as permanent fixtures. The
dependence measure is the best example: Chatterjee's ξ breaks x-ties
uniformly at random for a reason — a fixed ordering correlates with row
position, and the regression tests pin the failure exactly. A block-id
column against a row counter must report ξ ≈ 0, not the ≈ 0.99 a
stable-order implementation produced; tied x against independent y
must not inflate; a step function in y must still score high. Each of
those is a planted table and an assertion on the number.

The same discipline covers the sampler (strata coverage, no periodic
aliasing), the FD screen (near-unique determinants excluded), the
identity probe (violations counted, tolerance honored), and the rule
evaluator (thresholds inclusive of their boundaries).

## Python round-trip and notebooks

The Python suite runs the compiled wheel end-to-end: headline shape,
finding categories, coverage statuses, determinism of the serialized
report, the detector-selection contract, and the schema-version field.
The example notebooks are not decoration — CI executes
`onceover_analysis.ipynb` cell by cell against the freshly built wheel
and regenerates its stored outputs, so the documented report and the
shipped report cannot drift apart. The book chapters you are reading
execute the same way during the documentation build.

## The scale gate

Before any publication, a reproducible benchmark profiles a wide, known
table: 2,000,000 rows × 200 columns, fixed seed, independent noise, an
exact integer key, a deliberately dirty key, and three relationships
planted in the *last* columns — an affine identity, a nonlinear response,
and a code-to-label dependency — where a screener that only examined
the first few dozen columns would miss them.

The script fails unless every planted structure is recovered and the
exact cardinality checks hold. The recorded run, on macOS arm64 with
four Rayon threads:

| Measurement | Value |
|---|---:|
| Profiling time | 5.78 s |
| Peak process RSS during profiling | 3.23 GiB |
| Increase from pre-profile RSS | 0.16 GiB |

All eight checks pass, including recovery of the late-column
relationships, and no independent noise target is flagged as
predictable — the false-positive control. A smaller regression subset
runs in CI on every push; the full gate is the pre-publish step, with
the methodology and the recorded JSON in the
[benchmark notes](https://github.com/fcela/onceover/blob/main/benchmarks/README.md).

The honest summary of all of it: the tests do not prove the profiler is
right about your table. They prove it is right about tables built to
make it wrong, which is the part that can be engineered.